In [1]:
import ee

# Inicializar GEE
ee.Initialize()



In [1]:
import ee
import time

ee.Initialize()

anos = [2001, 2006, 2011, 2016]

# Geometria de Moçambique
moz = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
    .filter(ee.Filter.eq("country_na", "Mozambique"))
moz_geom = moz.geometry()

# Legenda oficial MODIS LC_Type1
legenda = {
    0: "No Data",
    1: "Evergreen Needleleaf Forest",
    2: "Evergreen Broadleaf Forest",
    3: "Deciduous Needleleaf Forest",
    4: "Deciduous Broadleaf Forest",
    5: "Mixed Forests",
    6: "Closed Shrublands",
    7: "Open Shrublands",
    8: "Woody Savannas",
    9: "Savannas",
    10: "Grasslands",
    11: "Permanent Wetlands",
    12: "Croplands",
    13: "Urban and Built-up Lands",
    14: "Cropland/Natural Vegetation Mosaic",
    15: "Snow and Ice",
    16: "Barren or Sparsely Vegetated",
    17: "Water Bodies"
}

# Exportar legenda como FeatureCollection para Drive
leg_fc = ee.FeatureCollection([
    ee.Feature(None, {"class": k, "description": v}) for k, v in legenda.items()
])

task_legenda = ee.batch.Export.table.toDrive(
    collection=leg_fc,
    description='Legenda_MODIS_LC_Type1',
    folder='GEE_MODIS_LC',
    fileNamePrefix='legenda_modis_lc_type1',
    fileFormat='CSV'
)
task_legenda.start()
print("Exportação da legenda iniciada.")

# Exporta imagens MODIS LC_Type1 para cada ano
for ano in anos:
    print(f"Processando ano {ano}...")
    modis_img = ee.ImageCollection("MODIS/061/MCD12Q1") \
        .filterDate(f'{ano}-01-01', f'{ano}-12-31') \
        .first()

    modis = ee.Algorithms.If(
        modis_img,
        modis_img.select('LC_Type1').clip(moz_geom),
        None
    )

    if modis is None:
        print(f"Imagem não encontrada para {ano}.")
        continue

    export = ee.batch.Export.image.toDrive(
        image=ee.Image(modis).toInt8(),
        description=f'MODIS_LC_Type1_{ano}',
        folder='GEE_MODIS_LC',
        fileNamePrefix=f'MCD12Q1_LC_Type1_{ano}_Moz_v061',
        region=moz_geom,
        scale=500,
        maxPixels=1e13,
        fileFormat='GeoTIFF'

    )

    export.start()
    print(f"Exportação iniciada para o ano {ano}.")

    # Acompanhar status da tarefa
    while export.active():
        print(f"Aguardando conclusão da exportação de {ano}...")
        time.sleep(30)

print("Todas as exportações finalizadas.")


Exportação da legenda iniciada.
Processando ano 2001...
Exportação iniciada para o ano 2001.
Aguardando conclusão da exportação de 2001...
Aguardando conclusão da exportação de 2001...
Aguardando conclusão da exportação de 2001...
Aguardando conclusão da exportação de 2001...
Aguardando conclusão da exportação de 2001...
Aguardando conclusão da exportação de 2001...
Aguardando conclusão da exportação de 2001...
Aguardando conclusão da exportação de 2001...
Aguardando conclusão da exportação de 2001...
Processando ano 2006...
Exportação iniciada para o ano 2006.
Aguardando conclusão da exportação de 2006...
Aguardando conclusão da exportação de 2006...
Aguardando conclusão da exportação de 2006...
Aguardando conclusão da exportação de 2006...
Aguardando conclusão da exportação de 2006...
Aguardando conclusão da exportação de 2006...
Processando ano 2011...
Exportação iniciada para o ano 2011.
Aguardando conclusão da exportação de 2011...
Aguardando conclusão da exportação de 2011...
Agu

In [2]:
# Lista de anos desejados
anos = [2001, 2006, 2011, 2016]

# Geometria de Moçambique
moz = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
    .filter(ee.Filter.eq("country_na", "Mozambique")) \
    .geometry()

# Função para processar e exportar a imagem com atributos completos
def exportar_landsat_lulc_completo(ano):
    print(f"Processando ano: {ano}")

    # Seleção da coleção apropriada
    colecao = "LANDSAT/LT05/C02/T1_L2" if ano <= 2011 else "LANDSAT/LE07/C02/T1_L2"

    # Correção radiométrica + recorte + filtragem
    imagem = ee.ImageCollection(colecao) \
        .filterDate(f'{ano}-01-01', f'{ano}-12-31') \
        .filterBounds(moz) \
        .filter(ee.Filter.lt('CLOUD_COVER', 30)) \
        .median() \
        .clip(moz) \
        .multiply(0.0000275).add(-0.2)

    # Índices espectrais
    ndvi = imagem.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')
    ndwi = imagem.normalizedDifference(['SR_B2', 'SR_B4']).rename('NDWI')
    ndbi = imagem.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDBI')

    # Topografia
    dem = ee.Image("USGS/SRTMGL1_003").clip(moz)
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem).rename('aspect')
    elevation = dem.rename('elevation')

    # Textura GLCM usando banda NIR (SR_B4)
    sr_b4_scaled = imagem.select('SR_B4').multiply(1000).toInt()
    glcm = sr_b4_scaled.glcmTexture(size=3)

    # Seleciona métricas GLCM úteis
    texturas = glcm.select([
        'SR_B4_contrast', 'SR_B4_diss', 'SR_B4_ent', 'SR_B4_asm',
        'SR_B4_corr', 'SR_B4_idm', 'SR_B4_savg', 'SR_B4_var'
    ])

    # Combina todos os atributos
    imagem_final = imagem.select([
        'SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'
    ]).addBands([
        ndvi, ndwi, ndbi, elevation, slope, aspect
    ]).addBands(texturas)

    # Exporta para o Google Drive
    export = ee.batch.Export.image.toDrive(
        image=imagem_final.toFloat(),
        description=f'Landsat_LULC_Full_{ano}',
        folder='GEE_LULC_FULL',
        fileNamePrefix=f'Landsat_LULC_{ano}_Moz_full',
        region=moz,
        scale=30,
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    export.start()
    print(f"Exportação iniciada para {ano}")

# Executa para cada ano
for ano in anos:
    exportar_landsat_lulc_completo(ano)


Processando ano: 2001
Exportação iniciada para 2001
Processando ano: 2006
Exportação iniciada para 2006
Processando ano: 2011
Exportação iniciada para 2011
Processando ano: 2016
Exportação iniciada para 2016


In [2]:
import ee
import time

ee.Initialize()

# =============================
# 1. Parâmetros
# =============================
anos = [2001, 2006, 2011, 2016]

# Região: Moçambique
moz = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
        .filter(ee.Filter.eq('country_na', 'Mozambique')) \
        .geometry()

# =============================
# 2. Aguarda tarefa terminar
# =============================
def aguardar_exportacao(task, ano):
    print(f"⏳ Aguardando exportação do ano {ano}...")
    while task.active():
        print(f"🔄 Exportando {ano}... aguardando...")
        time.sleep(30)
    print(f"✅ Exportação finalizada para o ano {ano}. Status: {task.status()['state']}")
    print("—" * 50)

# =============================
# 3. Função para processar ano
# =============================
def processar_ano(ano):
    start_date = f'{ano}-01-01'
    end_date   = f'{ano}-12-31'

    # MODIS Surface Reflectance
    modis_sr = ee.ImageCollection("MODIS/061/MOD09GA") \
        .filterDate(start_date, end_date) \
        .filterBounds(moz) \
        .select(['sur_refl_b01', 'sur_refl_b02', 'sur_refl_b06']) \
        .median().multiply(0.0001).clip(moz)

    red  = modis_sr.select('sur_refl_b01')
    nir  = modis_sr.select('sur_refl_b02')
    swir = modis_sr.select('sur_refl_b06')

    # Índices espectrais
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')
    ndwi = red.subtract(swir).divide(red.add(swir)).rename('NDWI')
    ndbi = swir.subtract(nir).divide(swir.add(nir)).rename('NDBI')

    # LST
    lst = ee.ImageCollection("MODIS/061/MOD11A2") \
        .filterDate(start_date, end_date) \
        .select('LST_Day_1km') \
        .mean().multiply(0.02).subtract(273.15).rename('LST') \
        .clip(moz)

    # Albedo
    albedo = ee.ImageCollection("MODIS/061/MCD43A3") \
        .filterDate(start_date, end_date) \
        .select('Albedo_WSA_shortwave') \
        .mean().rename('Albedo') \
        .clip(moz)

    # Topografia
    dem = ee.Image("CGIAR/SRTM90_V4").clip(moz)
    elevation = dem.rename('Elevation')
    slope = ee.Terrain.slope(dem).rename('Slope')
    aspect = ee.Terrain.aspect(dem).rename('Aspect')

    # Entropia GLCM
    nir_scaled = nir.multiply(1000).toInt()
    glcm = nir_scaled.glcmTexture()
    entropy = glcm.select('sur_refl_b02_ent').rename('Entropy')

    # Empilha tudo
    stack = ndvi.addBands([
        ndwi, ndbi, lst, albedo,
        elevation, slope, aspect, entropy
    ])

    # Exporta
    task = ee.batch.Export.image.toDrive(
        image=stack,
        description=f'MODIS_INDICES_{ano}',
        folder='GEE_EXPORT',
        fileNamePrefix=f'MODIS_Indices_MOZ_{ano}',
        region=moz,
        scale=500,
        crs='EPSG:4326',
        maxPixels=1e13
    )

    task.start()
    aguardar_exportacao(task, ano)

# =============================
# 4. Processa todos os anos
# =============================
for ano in anos:
    processar_ano(ano)

print("🎉 Todas as exportações foram concluídas com sucesso!")


⏳ Aguardando exportação do ano 2001...
🔄 Exportando 2001... aguardando...
✅ Exportação finalizada para o ano 2001. Status: FAILED
——————————————————————————————————————————————————
⏳ Aguardando exportação do ano 2006...
🔄 Exportando 2006... aguardando...
✅ Exportação finalizada para o ano 2006. Status: FAILED
——————————————————————————————————————————————————
⏳ Aguardando exportação do ano 2011...
🔄 Exportando 2011... aguardando...
✅ Exportação finalizada para o ano 2011. Status: FAILED
——————————————————————————————————————————————————
⏳ Aguardando exportação do ano 2016...
🔄 Exportando 2016... aguardando...
✅ Exportação finalizada para o ano 2016. Status: FAILED
——————————————————————————————————————————————————
🎉 Todas as exportações foram concluídas com sucesso!


In [1]:
!pip install geemap ee

  Using cached ee-0.2.tar.gz (3.0 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached blessings-1.7-py3-none-any.whl.metadata (19 kB)
Using cached blessings-1.7-py3-none-any.whl (18 kB)
  Created wheel for ee: filename=ee-0.2-py3-none-any.whl size=3706 sha256=d14ab269a3e49997f8ac7be710a434391b9e81e4c9fa4e3bb8aa978aedcc77f5
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\7c\e1\02\eaee729f44caf802f4f44282b79f66bcb8559ccfd405a0c6c4
Successfully built ee



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import ee
import geemap

ee.Authenticate()
ee.Initialize()

# Geometria de Moçambique
countries = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0')
mozambique = countries.filter(ee.Filter.eq('ADM0_NAME', 'Mozambique')).geometry()

# Anos desejados
anos = [2001, 2006, 2011, 2016, 2021]

for ano in anos:
    print(f"\n📦 Processando ano {ano}...")

    # Escolher Landsat correto
    if ano <= 2011:
        landsat_col = 'LANDSAT/LT05/C02/T1_L2'
    elif ano <= 2013:
        landsat_col = 'LANDSAT/LE07/C02/T1_L2'
    else:
        landsat_col = 'LANDSAT/LC08/C02/T1_L2'

    col = ee.ImageCollection(landsat_col) \
        .filterDate(f'{ano}-01-01', f'{ano}-12-31') \
        .filterBounds(mozambique)

    # Projeção de referência
    ref_image = col.first()
    projRef = ref_image.select(0).projection()

    # Função para preparar as bandas e índices
    def preparar(imagem):
        bandas = imagem.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6']) \
            .rename(['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1']) \
            .multiply(0.0000275).add(-0.2)

        ndvi = bandas.normalizedDifference(['NIR', 'RED']).rename('NDVI')
        evi = bandas.expression(
            '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
            {'NIR': bandas.select('NIR'), 'RED': bandas.select('RED'), 'BLUE': bandas.select('BLUE')}
        ).rename('EVI')
        ndmi = bandas.normalizedDifference(['NIR', 'SWIR1']).rename('NDMI')
        ndbi = bandas.normalizedDifference(['SWIR1', 'NIR']).rename('NDBI')
        savi = bandas.expression(
            '((NIR - RED) / (NIR + RED + 0.5)) * 1.5',
            {'NIR': bandas.select('NIR'), 'RED': bandas.select('RED')}
        ).rename('SAVI')

        return bandas.addBands([ndvi, evi, ndmi, ndbi, savi])

    col_indices = col.map(preparar).median()

    # Tentar adicionar LST se disponível
    try:
        lst = col.select('ST_B10') \
            .map(lambda img: img.multiply(0.00341802).add(149).subtract(273.15).rename('LST')) \
            .median()
        stack = col_indices.addBands(lst)
    except:
        print("⚠️ LST não disponível para este sensor, pulando...")
        stack = col_indices

    # Recorte para Moçambique
    final = stack.clip(mozambique)

    # Exportar imagem
    out_path = f"Landsat_Only_Stack_{ano}.tif"
    geemap.ee_export_image(
        image=final,
        filename=out_path,
        region=mozambique,
        scale=30,
        crs='EPSG:4326',
        file_per_band=False
    )

    print(f"✅ Imagem {ano} salva como {out_path}")


ModuleNotFoundError: No module named '_curses'